# Pakistan House Price Predictor: Data Cleaning and EDA

**Project 1** | Goal: predict house sale prices in Pakistani cities.

**Notebook 01:** load the data, clean it, and save a cleaned file for modelling.

## 1. Dataset Information

- **Source:** [Pakistan House Price Dataset (Kaggle)](https://www.kaggle.com/datasets/jillanisofttech/pakistan-house-price-dataset), based on Zameen.com listings
- **Size:** 168,446 rows and 20 columns
- **Data types:**
  - int64 (5): property_id, location_id, price, baths, bedrooms
  - float64 (3): latitude, longitude, Area Size
  - object (12): page_url, property_type, location, city, province_name, area, purpose, date_added, agency, agent, Area Type, Area Category
- **Missing values:** `agency` has 44,071 (about 26%) and `agent` has 44,072 (about 26%). All other columns have no NaN values.

### Column descriptions

| Column | Meaning | Type |
|---|---|---|
| property_id | Unique ID of the listing | int |
| location_id | ID of the locality | int |
| page_url | Link to the Zameen listing | object |
| property_type | House, Flat, Upper Portion, Lower Portion, Room, Farm House, Penthouse | object |
| price | Listed price in PKR | int |
| location | Neighbourhood or society | object |
| city | City | object |
| province_name | Province | object |
| latitude, longitude | Map coordinates | float |
| baths | Number of bathrooms | int |
| area | Area as text | object |
| purpose | For Sale or For Rent | object |
| bedrooms | Number of bedrooms | int |
| date_added | Date the listing was posted (month-day-year) | object |
| agency, agent | Seller's agency and agent | object |
| Area Type | Unit of the area: Marla or Kanal | object |
| Area Size | Area value in the unit given by Area Type | float |
| Area Category | Size bucket such as "0-5 Marla" | object |

## 2. Load the data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/zameen-updated.csv")
rows_before = len(df)          # starting size, used in the final report
summary = {}                   # rows removed at each cleaning step

print("Data shape:", df.shape)
df.info()

Data shape: (168446, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168446 entries, 0 to 168445
Data columns (total 20 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   property_id    168446 non-null  int64  
 1   location_id    168446 non-null  int64  
 2   page_url       168446 non-null  object 
 3   property_type  168446 non-null  object 
 4   price          168446 non-null  int64  
 5   location       168446 non-null  object 
 6   city           168446 non-null  object 
 7   province_name  168446 non-null  object 
 8   latitude       168446 non-null  float64
 9   longitude      168446 non-null  float64
 10  baths          168446 non-null  int64  
 11  area           168446 non-null  object 
 12  purpose        168446 non-null  object 
 13  bedrooms       168446 non-null  int64  
 14  date_added     168446 non-null  object 
 15  agency         124375 non-null  object 
 16  agent          124374 non-null  object 
 17  Area

In [2]:
df.isnull().sum()

property_id          0
location_id          0
page_url             0
property_type        0
price                0
location             0
city                 0
province_name        0
latitude             0
longitude            0
baths                0
area                 0
purpose              0
bedrooms             0
date_added           0
agency           44071
agent            44072
Area Type            0
Area Size            0
Area Category        0
dtype: int64

In [3]:
df.head()

,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent,Area Type,Area Size,Area Category
0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,For Sale,2,02-04-2019,NaN,NaN,Marla,4.0,0-5 Marla
1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,For Sale,3,05-04-2019,NaN,NaN,Marla,5.6,5-10 Marla
2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,For Sale,5,07-17-2019,NaN,NaN,Marla,8.0,5-10 Marla
3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,For Sale,4,04-05-2019,NaN,NaN,Kanal,2.0,1-5 Kanal
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,Marla,8.0,5-10 Marla


## 3. Initial checks

Before changing anything, I check three things that can silently ruin a price model: mixed purposes (sale vs rent), mixed area units, and duplicate listings.

In [4]:
df["purpose"].value_counts()

purpose
For Sale    120655
For Rent     47791
Name: count, dtype: int64

In [5]:
df["Area Type"].value_counts()

Area Type
Marla    137769
Kanal     30677
Name: count, dtype: int64

In [6]:
df["property_type"].value_counts()

property_type
House            105468
Flat              38238
Upper Portion     13774
Lower Portion      9229
Room                685
Farm House          657
Penthouse           395
Name: count, dtype: int64

In [7]:
# Every listing has a unique property_id, so a plain df.duplicated() always returns 0.
# A stricter check ignores the ID and the URL.
print("Plain duplicates:", df.duplicated().sum())
print("Duplicates ignoring ID and URL:", df.drop(columns=["property_id", "page_url"]).duplicated().sum())

Plain duplicates: 0
Duplicates ignoring ID and URL: 15011


## 4. Keep only "For Sale" listings

Rent is a monthly amount and sale price is a total amount. Mixing them in one `price` column would confuse the model, and I want to predict sale prices, so I keep only "For Sale".

In [8]:
before = len(df)
df = df[df["purpose"] == "For Sale"].copy()
summary["Rental listings removed"] = before - len(df)
print("Rows removed:", before - len(df), "| Rows now:", len(df))

Rows removed: 47791 | Rows now: 120655


## 5. Convert all areas to Marla

Areas come in two units. **1 Kanal = 20 Marla.** `Area Size` is the number in the unit given by `Area Type`, so a Kanal row must be multiplied by 20.

In [9]:
df["area_marla"] = df["Area Size"].where(df["Area Type"] == "Marla", df["Area Size"] * 20)

In [10]:
# Verification: Kanal rows must be exactly 20 times Area Size, Marla rows must be unchanged
kanal = df["Area Type"] == "Kanal"
print("Kanal rows correct:", (df.loc[kanal, "area_marla"] == df.loc[kanal, "Area Size"] * 20).all())
print("Marla rows correct:", (df.loc[~kanal, "area_marla"] == df.loc[~kanal, "Area Size"]).all())

Kanal rows correct: True
Marla rows correct: True


In [11]:
df["area_marla"].describe()

count    120655.000000
mean         11.065215
std          59.560442
min           0.000000
25%           4.800000
50%           6.400000
75%          10.000000
max       12000.000000
Name: area_marla, dtype: float64

## 6. Convert `date_added` to a real date

The column is text. I parse it as month-day-year (for example `07-17-2019`) and add year and month columns.

In [12]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m-%d-%Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["date_added"].dtype

dtype('<M8[ns]')

## 7. Hidden missing values: zero bedrooms and baths

`isnull()` shows no missing values in `bedrooms` and `baths`, but many rows contain 0. A house or flat with 0 bathrooms is not realistic, so 0 most likely means "not entered". I will treat these zeros as missing values later in this notebook.

In [13]:
print("Bedrooms = 0:", (df["bedrooms"] == 0).sum())
print("Baths = 0:", (df["baths"] == 0).sum())

Bedrooms = 0: 13119
Baths = 0: 27554


In [14]:
df[df["bedrooms"] == 0]["property_type"].value_counts()

property_type
House            9969
Flat             2295
Farm House        371
Upper Portion     323
Lower Portion     115
Penthouse          39
Room                7
Name: count, dtype: int64

In [15]:
df[df["baths"] == 0]["property_type"].value_counts()

property_type
House            20285
Flat              5774
Upper Portion      701
Farm House         403
Lower Portion      299
Penthouse           82
Room                10
Name: count, dtype: int64

## 8. Drop columns that do not help predict price

- `area`, `Area Type`, `Area Size`, `Area Category`: all replaced by `area_marla` (checked below that Area Category is only a size bucket)
- `agency`, `agent`, `page_url`: describe the seller or the listing, not the house (and 26% of agent and agency values are missing)
- `property_id`: an ID, no predictive value
- `purpose`: now the same value ("For Sale") in every row

In [16]:
df["Area Category"].value_counts().head(10)

Area Category
0-5 Marla      51214
5-10 Marla     40608
1-5 Kanal      18000
10-15 Marla     8201
15-20 Marla     2135
5-10 Kanal       289
15-20 Kanal       78
10-15 Kanal       40
20-30 Kanal       36
30-40 Kanal       21
Name: count, dtype: int64

In [17]:
cols_to_drop = ["area", "Area Type", "Area Size", "Area Category",
                "agency", "agent", "page_url", "purpose", "property_id"]
df = df.drop(columns=cols_to_drop)   # drop returns a new DataFrame, so it must be assigned back
df.columns.tolist()

['location_id',
 'property_type',
 'price',
 'location',
 'city',
 'province_name',
 'latitude',
 'longitude',
 'baths',
 'bedrooms',
 'date_added',
 'area_marla',
 'year_added',
 'month_added']

## 9. Remove duplicate listings

The same house can be posted several times under different IDs. Duplicates inflate the data and can leak the same house into both the training and test sets, which makes accuracy look better than it is.

I compare every column except the dates, because a reposted house gets a new date.

In [18]:
dup_cols = [c for c in df.columns if c not in ["date_added", "year_added", "month_added"]]
print("Duplicates found:", df.duplicated(subset=dup_cols).sum())

# Look at a sample before deciding
dups = df[df.duplicated(subset=dup_cols, keep=False)].sort_values(dup_cols)
dups[["property_type", "location", "price", "area_marla", "bedrooms", "baths", "latitude", "longitude"]].head(20)

Duplicates found: 38286


,property_type,location,price,area_marla,bedrooms,baths,latitude,longitude
122494,House,Garden Town,11500000,5.0,3,3,31.503922,74.323139
122495,House,Garden Town,11500000,5.0,3,3,31.503922,74.323139
103502,House,Garden Town,12500000,5.0,3,3,31.503922,74.323139
103503,House,Garden Town,12500000,5.0,3,3,31.503922,74.323139
122496,House,Garden Town,12500000,5.0,3,3,31.503922,74.323139
103504,House,Garden Town,15000000,5.0,3,3,31.503922,74.323139
103547,House,Garden Town,15000000,5.0,3,3,31.503922,74.323139
122497,House,Garden Town,15000000,5.0,3,3,31.503922,74.323139
122498,House,Garden Town,15000000,5.0,3,3,31.503922,74.323139
68858,House,Garden Town,20000000,7.0,4,4,31.503922,74.323139


In [19]:
before = len(df)
df = df.drop_duplicates(subset=dup_cols).copy()
summary["Duplicate listings removed"] = before - len(df)
print("Removed:", before - len(df), "| Rows now:", len(df))

Removed: 38286 | Rows now: 82369


## 10. Area outliers

`area_marla` has a minimum of 0 (impossible) and a maximum of 12,000 marla (600 Kanal), while the median is only about 6 marla. A few extreme values pull the mean far above the median.

**Rule:** keep houses between 1 and 100 marla (up to 5 Kanal), which covers normal residential properties.

In [20]:
print("Zero area:", (df["area_marla"] == 0).sum())
print(df["area_marla"].quantile([0.01, 0.99, 0.999]))

Zero area: 9
0.010      1.800
0.990     80.000
0.999    416.432
Name: area_marla, dtype: float64


In [21]:
df.sort_values("area_marla", ascending=False)[["property_type", "city", "price", "area_marla"]].head(10)

,property_type,city,price,area_marla
68989,Lower Portion,Karachi,2200000,12000.0
7711,Farm House,Lahore,884000000,8320.0
74447,Farm House,Islamabad,450000000,6000.0
151417,Farm House,Lahore,80000000,5100.0
43783,Farm House,Islamabad,300000000,3360.0
45173,Farm House,Islamabad,300000000,3360.0
149501,Flat,Islamabad,15000000,2302.0
4891,Farm House,Islamabad,350000000,2000.0
103979,Farm House,Lahore,200000000,2000.0
90100,House,Lahore,220000000,1800.0


In [22]:
mask = df["area_marla"].between(1, 100)
print("Rows removed:", (~mask).sum())
before = len(df)
df = df[mask].copy()
summary["Area outliers removed"] = before - len(df)
df["area_marla"].describe()

Rows removed: 486


count    81883.000000
mean         9.776786
std          9.769319
min          1.000000
25%          4.700000
50%          6.400000
75%         10.000000
max        100.000000
Name: area_marla, dtype: float64

## 11. Fill the zero bedrooms and baths

Deleting every row with a zero would throw away a large part of the data. Instead I turn the zeros into NaN, then fill them with the **median of the same property type**. The median is not affected by extreme values, and grouping by property type is fair because a flat and a farm house have different typical values. A flag column records which values were filled in.

In [23]:
for col in ["bedrooms", "baths"]:
    df[col + "_was_missing"] = (df[col] == 0)
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df.groupby("property_type")[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())      # safety net for a group with no valid value
    df[col] = df[col].round().astype(int)

df[["bedrooms", "baths"]].describe()

,bedrooms,baths
count,81883.000000,81883.000000
mean,3.886326,4.113113
std,1.636303,2.223120
min,1.000000,1.000000
25%,3.000000,3.000000
50%,4.000000,4.000000
75%,5.000000,5.000000
max,27.000000,403.000000


In [24]:
df[["bedrooms", "baths"]].isnull().sum()

bedrooms    0
baths       0
dtype: int64

## 12. Price outliers

Prices that are impossibly low or high would pull the model in the wrong direction. First I look at the distribution and the extremes.

In [25]:
df["price"].describe()

count    8.188300e+04
mean     2.527889e+07
std      3.813089e+07
min      0.000000e+00
25%      7.000000e+06
50%      1.360000e+07
75%      2.700000e+07
max      1.000000e+09
Name: price, dtype: float64

In [26]:
df.sort_values("price")[["property_type", "city", "area_marla", "price"]].head(10)

,property_type,city,area_marla,price
132855,House,Rawalpindi,10.0,0
167202,Flat,Karachi,9.6,1
74134,House,Karachi,20.0,1
156148,House,Karachi,4.8,1
156361,House,Rawalpindi,10.0,1
9808,House,Lahore,5.0,1
162480,House,Rawalpindi,5.0,1
102235,House,Rawalpindi,12.0,3
35917,House,Karachi,15.0,4
9809,House,Lahore,20.0,4


In [27]:
df.sort_values("price", ascending=False)[["property_type", "city", "area_marla", "price"]].head(10)

,property_type,city,area_marla,price
55589,House,Karachi,80.0,1000000000
95179,House,Lahore,20.0,935000000
379,House,Islamabad,96.0,850000000
11418,House,Islamabad,96.0,850000000
102164,House,Lahore,40.0,805000000
37650,House,Lahore,100.0,750000000
126286,House,Islamabad,74.0,700000000
43174,House,Karachi,8.0,690000000
5742,House,Rawalpindi,42.0,650000000
69350,House,Lahore,92.0,650000000


**Rule:** keep prices between the 1st and 99th percentile. (After running, check that the low value is believable for a house in Pakistan and the high value is believable for a large property.)

In [28]:
low, high = df["price"].quantile([0.01, 0.99])
print(f"Low: {low:,.0f}  High: {high:,.0f}")
print("Rows removed:", (~df["price"].between(low, high)).sum())

before = len(df)
df = df[df["price"].between(low, high)].copy()
summary["Price outliers removed"] = before - len(df)

Low: 1,800,000  High: 190,000,000
Rows removed: 1532


## 13. Final report and save

Final check of the cleaned data: how many rows were removed at each step, whether any values are missing, and what the table looks like.

In [29]:
df = df.reset_index(drop=True)

report = pd.Series(summary, name="rows removed").to_frame()
report.loc["TOTAL removed"] = rows_before - len(df)
display(report)

print("Rows before:", rows_before, "| Rows after:", len(df))
print(f"Kept: {len(df) / rows_before:.1%} of the original data")
print("Total missing values:", df.isnull().sum().sum())

,rows removed
Rental listings removed,47791
Duplicate listings removed,38286
Area outliers removed,486
Price outliers removed,1532
TOTAL removed,88095


Rows before: 168446 | Rows after: 80351
Kept: 47.7% of the original data
Total missing values: 0


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80351 entries, 0 to 80350
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   location_id           80351 non-null  int64         
 1   property_type         80351 non-null  object        
 2   price                 80351 non-null  int64         
 3   location              80351 non-null  object        
 4   city                  80351 non-null  object        
 5   province_name         80351 non-null  object        
 6   latitude              80351 non-null  float64       
 7   longitude             80351 non-null  float64       
 8   baths                 80351 non-null  int64         
 9   bedrooms              80351 non-null  int64         
 10  date_added            80351 non-null  datetime64[ns]
 11  area_marla            80351 non-null  float64       
 12  year_added            80351 non-null  int32         
 13  month_added     

In [31]:
df.head()

,location_id,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,area_marla,year_added,month_added,bedrooms_was_missing,baths_was_missing
0,3325,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2019-02-04,4.0,2019,2,False,False
1,3236,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,2019-05-04,5.6,2019,5,False,False
2,764,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,2019-07-17,8.0,2019,7,False,False
3,340,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,2019-04-05,40.0,2019,4,False,False
4,3226,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,2019-07-10,8.0,2019,7,False,False


In [32]:
df.to_csv("../data/cleaned_house_prices.csv", index=False)
print("Saved:", df.shape)

Saved: (80351, 16)


## What I learned in this notebook

1. **Rent and sale prices cannot share one column.** They mean different things, so I kept only "For Sale".
2. **A number needs its unit.** `Area Size` is only meaningful together with `Area Type`. I converted everything to Marla (1 Kanal = 20 Marla) and verified the result.
3. **`df.drop()` returns a new DataFrame.** It must be assigned back: `df = df.drop(...)`.
4. **Missing values can hide as zeros.** `isnull()` reported none in bedrooms, but 0 was used for "not entered".
5. **A unique ID hides duplicates.** `df.duplicated()` returned 0 until I ignored `property_id`.
6. **After filtering, row numbers stay the same.** A `KeyError` from `loc` can mean the row was filtered out, not that the code is wrong.
7. **Run order matters.** I finish with Restart kernel and Run All so the notebook can be reproduced from top to bottom.